# **Part 1: Install libraries, load the Qwen model, load the PDF, split it, create embeddings, and build the Chroma database.**

In [59]:
# Step 1: Install Libraries

!pip install -q \
langchain \
langgraph \
langchain-community \
langchain-huggingface \
langchain-chroma \
langchain-text-splitters \
sentence-transformers \
transformers \
torch \
accelerate \
chromadb \
pypdf \
beautifulsoup4 \
requests

In [60]:
# Step 2: Import Libraries

import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_huggingface import (
    HuggingFacePipeline,
    ChatHuggingFace,
    HuggingFaceEmbeddings
)

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

In [61]:
# Step 3: Prevent TensorFlow Issues

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [62]:
# Step 4: Upload Your Syllabus PDF

from google.colab import files

uploaded = files.upload()

Saving syllabus.pdf to syllabus (3).pdf


In [63]:
# Step 5: Load the Local Qwen Model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [64]:
# Step 6: Create the Text Generation Pipeline

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False
)

In [65]:
# Step 7: Convert it into a LangChain Chat Model

llm = ChatHuggingFace(
    llm=HuggingFacePipeline(
        pipeline=text_pipeline
    )
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [66]:
# Step 8: Load the PDF

loader = PyPDFLoader("syllabus (1).pdf")

documents = loader.load()

print("Pages Loaded:", len(documents))

Pages Loaded: 21


In [67]:
# Step 9: Split the PDF into Chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 27


In [68]:
# Step 10: Create Embeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [69]:
# Step 11: Create the Chroma Database

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db",
    collection_name="techolas_syllabus"
)

print("Vector Database Created")

Vector Database Created


In [70]:
# Step 12: Create the Retriever

retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 3
    }
)

In [71]:
# Step 13: Test Retrieval

query = "What are the prerequisites for this course?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print("=" * 50)
    print(f"Result {i}")
    print(doc.page_content)

Result 1
Industry-Recognized Credentials
Your skills are validated by leading certification
bodies like ISO, IAF, and NC/uni0422/uni0422.
Certified For Success
SAMPLE
Result 2
Industry-Recognized Credentials
Your skills are validated by leading certification
bodies like ISO, IAF, and NC/uni0422/uni0422.
Certified For Success
SAMPLE
Result 3
Industry-Recognized Credentials
Your skills are validated by leading certification
bodies like ISO, IAF, and NC/uni0422/uni0422.
Certified For Success
SAMPLE


In [72]:
# What you've built
#                 syllabus.pdf
#                      │
#                      ▼
#               PyPDFLoader
#                      │
#                      ▼
#       RecursiveCharacterTextSplitter
#                      │
#                      ▼
#         HuggingFace Embeddings
#                      │
#                      ▼
#              Chroma Vector DB
#                      │
#                      ▼
#                Retriever Ready

# **Part 2: Build the RAG retriever and answer from the syllabus.**

In [73]:
# Step 1: Create a Helper Function

def format_docs(docs):
    """
    Combine retrieved documents into one text block.
    """
    return "\n\n".join(doc.page_content for doc in docs)

In [74]:
# Step 2: Create the Prompt

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a Techolas syllabus assistant.

Answer ONLY using the syllabus context below.

If the answer is not present in the context, reply exactly:

"I don't know based on the syllabus."

Do NOT make up information.

--------------------
Context:
{context}
--------------------

Question:
{question}

Answer:
""")

In [75]:
#Step 3: Create the Output Parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [76]:
# Step 4: Build the RAG Chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser
)

In [77]:
# Step 5: Test the Chatbot

question = "What is the duration of the course?"

answer = rag_chain.invoke(question)

print(answer)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The course has 60 hours of pre-placement programs.


In [78]:
# Step 6: Ask More Questions

questions = [
    "Who can join this course?",
    "What projects are included?",
    "Is internship provided?",
    "What programming languages are taught?"
]

for q in questions:
    print("="*60)
    print("Question:", q)
    print()
    print(rag_chain.invoke(q))
    print()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Who can join this course?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This program is open to anyone who meets the eligibility criteria set forth in the syllabus. The eligibility requirements may vary depending on the specific program and location, but generally, participants must be at least 18 years old and have a valid driver's license or passport for international students. Additionally, they should have a basic understanding of computer skills such as typing, basic internet browsing, and email. It's important to note that some programs may have additional requirements or restrictions, so it's always best to check with the program directly for more detailed information.

Question: What projects are included?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The project that is included in this module is "Building a Strong Portfolio." This project focuses on validating analytical and technical coding skills through hosting end-to-end analytics projects, which highlight data cleaning, visualization, and insights. It also emphasizes building credibility within the analytics community and applying these skills to diverse business problems. The project demonstrates consistent collaboration, data analysis coding discipline, and well-documented Python or SQL scripts for analytics and reporting. Additionally, it highlights data cleaning techniques and showcases how these skills can be applied effectively in real-world scenarios.

Question: Is internship provided?



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, internship is provided in this program.

Question: What programming languages are taught?

Programming languages taught include Python, which is introduced in Module 2.



In [79]:
# Step 7: Test an Unknown Question

question = "Who won the FIFA World Cup?"

print(rag_chain.invoke(question))


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't know based on the syllabus.


In [80]:
# Step 8: Create a Simple Chat Function
def ask(question):
    answer = rag_chain.invoke(question)

    print("="*60)
    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("="*60)

ask("What is the course fee?")

ask("How many modules are there?")

ask("Is placement assistance available?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
What is the course fee?

Answer:
Based on the syllabus provided, there is no specific mention of a course fee for this program. Therefore, I don't know based on the syllabus.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
How many modules are there?

Answer:
There are 10 modules mentioned in the given syllabus: `Python Files and Directories Manipulation`, `Use Various Files and Directory Functions for OS Operations`, `Python Core Objects and Functions`, `Built-in-Modules (Library Functions)`, `Numeric and Math's Module`, `String/List/Dictionaries/Tuple`, `Complex Data Structure in Python`, `Arbitrary Data Types and Their Data Structure`, `Python Built-in Function`, `Python User-Defined Functions`, `Python Packages and Functions`, `Anonymous Functions - Lambda Functions`, `Object-Oriented Python`, `Oops Concepts`, `Object, Classes and Destroying Objects`, `Accessing Attributes, Built-in Class Attributes`, `Inheritance and Polymorphism`, `Overloading Operators`, `Exception Handling in Python`, `Handling Various Exceptions using try...except..else`, `Try-finally Clause`, and `The Argument of an Exception and Create`. Therefore, the number of modules is 10.
Question:
Is placement assistance availa

In [81]:
  #  What you've built

  #                User Question
  #                       │
  #                       ▼
  #                 Retriever
  #                       │
  #                       ▼
  #            Top 3 Relevant Chunks
  #                       │
  #                       ▼
  #                 Prompt Template
  #                       │
  #                       ▼
  #                    Qwen LLM
  #                       │
  #                       ▼
  #                String Parser
  #                       │
  #                       ▼
  #                    Final Answer

# **Part 3: Add the Techolas website fallback.**

In [82]:

# Step 1: Install BeautifulSoup
!pip install beautifulsoup4 requests

In [144]:
# Step 2: Import Libraries
from langchain_community.document_loaders import WebBaseLoader
# Step 3: Create a Website Loader
loader = WebBaseLoader("https://techolas.com")

web_docs = loader.load()

print(web_docs[0].page_content[:1000])






















Best Software Training Institute In Calicut








































Home
About Us
Courses
 AI Courses
Placements
Contact Us











Home
About Us
Courses
 AI Courses
Placements
Contact Us














Kerala's No. 1 Software Training Institute

Leaders in education since 2018, empowering minds and shaping the future.
Techolas Technologies Pvt Ltd : Leading IT Training Institute in  Calicut & Palakkad. Get job-ready with expert courses in Data Science, Data Analytics, Business Analytics, Python Full stack, MERN Stack, Data Analytics With GEN - AI and AI Integrated Courses 




                     Call Now











  Enquire Now












Our Courses
Trainings we offer
Our software training institute in Calicut and Palakkad offers diverse IT courses, including Data Science, Data Analytics, Web Development, Software Testing and Machine Learning . With hands-on projects and real-world case studies, Techolas Technologies ensures you gain the pract

In [145]:
# step 5: Split the Website into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

web_chunks = splitter.split_documents(web_docs)

print(len(web_chunks))

10


In [146]:
# Step 6: Create Website Chroma DB

website_db = Chroma.from_documents(
    documents=web_chunks,
    embedding=embeddings,
    collection_name="techolas_website"
)

In [147]:
# Step 7: Website Retriever

website_retriever = website_db.as_retriever(
    search_kwargs={"k":3}
)

In [148]:
# Step 8: Website Prompt

website_prompt = ChatPromptTemplate.from_template("""
You are a Techolas assistant.

Answer ONLY using the website context.

If the answer is not available reply exactly:

"I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

In [149]:
# Step 9: Website RAG Chain

website_chain = (
    {
        "context": website_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | website_prompt
    | llm
    | parser
)

In [150]:
text = web_docs[0].page_content.lower()

print("recent placements" in text)
print("placed" in text)
print("amazon" in text)
print("cognizant" in text)
print("tcs" in text)

True
True
False
False
False


In [91]:
# Step 10: Test It
question = "Where is Techolas located?"

answer = website_chain.invoke(question)

print(answer)

# Another example:

question = "Does Techolas provide placement assistance?"

print(website_chain.invoke(question))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Palakkad
Yes, Techolas provides placement assistance for its students.


In [92]:
# Step 11: Compare Both Chains

print(rag_chain.invoke("What is the course duration?"))
print(website_chain.invoke("What is the course duration?"))


print(rag_chain.invoke("Where is Techolas located?"))
print(website_chain.invoke("Where is Techolas located?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The program duration provided in the syllabus does not include any specific time frame for the courses. Therefore, I cannot provide an exact number without additional information. Based solely on the given syllabus, I am unable to determine the duration of the program.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Data Science: 8 Months
Data Analytics: 7 Months
Python Full Stack: 7 Months
MERN STACK: 7 Months
Business Analytics: 5 Months
Data Analysis with Gen-AI: 8 Months


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Techolas Technologies is located in China.
Palakkad


In [93]:
# Current Architecture
#                     User
#                      │
#                      ▼
#                 Question
#                      │
#          ┌───────────┴───────────┐
#          │                       │
#          ▼                       ▼
#    Syllabus Chroma         Website Chroma
#          │                       │
#          ▼                       ▼
#      PDF RAG Chain         Website RAG Chain
#          │                       │
#          └───────────┬───────────┘
#                      │
#                      ▼
#                  Final Answer

# **Part 4: Build the LangGraph workflow.**

In [151]:
# Step 1: Import LangGraph
from typing import TypedDict

from langgraph.graph import (
    StateGraph,
    START,
    END
)

In [152]:
# Step 2: Create the State
class ChatState(TypedDict):

    question: str

    pdf_answer: str

    website_answer: str

    final_answer: str

    source: str

In [153]:
# Step 3: PDF Search Node

def search_pdf(state):

    question = state["question"]

    answer = rag_chain.invoke(question)

    state["pdf_answer"] = answer

    return state

In [154]:
# Step 4: Router Function
def pdf_router(state):

    answer = state["pdf_answer"].lower()

    if (
        "i don't know" in answer
        or "do not have enough information" in answer
        or "not enough information" in answer
        or "cannot determine" in answer
        or "not mentioned" in answer
        or "cannot provide" in answer
    ):
        return "website"

    return "finish"

In [155]:
# Step 5: Website Search Node
def search_website(state):

    question = state["question"]

    answer = website_chain.invoke(question)

    state["website_answer"] = answer

    return state

In [156]:
# Step 6: Website Router
def website_router(state):

    answer = state["website_answer"].lower()

    if "i don't know" in answer:

        return "unknown"

    return "finish"

In [157]:
# Step 7: PDF Final Node
def pdf_final(state):

    state["final_answer"] = state["pdf_answer"]

    state["source"] = "PDF"

    return state

In [158]:
# Step 8: Website Final Node
def website_final(state):

    state["final_answer"] = state["website_answer"]

    state["source"] = "Website"

    return state

In [159]:
# Step 9: Unknown Node
def unknown(state):

    state["final_answer"] = (
        "I don't know. "
        "The information is not available in the syllabus or on the Techolas website."
    )

    state["source"] = "None"

    return state

In [160]:
# Step 10: Build the Graph
graph = StateGraph(ChatState)

graph.add_node("search_pdf", search_pdf)

graph.add_node("search_website", search_website)

graph.add_node("pdf_final", pdf_final)

graph.add_node("website_final", website_final)

graph.add_node("unknown", unknown)

In [161]:
# step 11: Add Edges
graph.add_edge(START, "search_pdf")

# first router

graph.add_conditional_edges(

    "search_pdf",

    pdf_router,

    {

        "finish": "pdf_final",

        "website": "search_website"

    }

)

# Website router
graph.add_conditional_edges(

    "search_website",

    website_router,

    {

        "finish": "website_final",

        "unknown": "unknown"

    }

)

# Finish nodes
graph.add_edge("pdf_final", END)

graph.add_edge("website_final", END)

graph.add_edge("unknown", END)

# Step 12: Compile
chatbot = graph.compile()

In [162]:
# Step 13: Test
result = chatbot.invoke({

    "question":"What is the duration of the course?"

})

print(result["final_answer"])

print(result["source"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The course has 60 hours of pre-placement programs.
PDF


In [164]:
result = chatbot.invoke({

    "question":"What is the recent placement at techolas?"

})

print(result["final_answer"])

print(result["source"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hanna Mariyam
Website


In [135]:
result = chatbot.invoke({

    "question":"Who won the FIFA World Cup 1998?"

})

print(result["final_answer"])

print(result["source"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The FIFA World Cup 1998 was won by Brazil.
PDF


In [109]:
  #               User
  #                 │
  #                 ▼
  #           search_pdf
  #                 │
  #     ┌───────────┴───────────┐
  #     │                       │
  #     ▼                       ▼
  # pdf_final            search_website
  #     │                       │
  #     ▼             ┌─────────┴─────────┐
  #    END            ▼                   ▼
  #            website_final          unknown
  #                 │                   │
  #                 ▼                   ▼
  #                END                 END

# **Part 5: Test the chatbot and refine it.**

In [110]:
# Step 1: Create a Chat Function

def ask(question):

    result = chatbot.invoke({
        "question": question
    })

    print("=" * 60)

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(result["final_answer"])

    print("\nSource:")
    print(result["source"])

    print("=" * 60)

    return result

In [111]:
ask("What is the duration of the course?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
What is the duration of the course?

Answer:
The course has 60 hours of pre-placement programs.

Source:
PDF


{'question': 'What is the duration of the course?',
 'pdf_answer': 'The course has 60 hours of pre-placement programs.',
 'final_answer': 'The course has 60 hours of pre-placement programs.',
 'source': 'PDF'}

In [112]:
# Step 2: Interactive Chatbot
print("=" * 60)
print("Techolas Syllabus Chatbot")
print("Type 'exit' to stop.")
print("=" * 60)

while True:

    question = input("\nYou : ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    result = chatbot.invoke({
        "question": question
    })

    print("\nBot :", result["final_answer"])
    print("Source :", result["source"])

Techolas Syllabus Chatbot
Type 'exit' to stop.

You : exit

Goodbye!
